# 🌪️ MARAHS: Multi-Agent Hurricane Coverage Training

## Research Contribution
First demonstration of **cooperative multi-drone coverage** in hurricane conditions using:
- **Graph Neural Network communication** between drones
- **Curriculum learning** from zero wind to Cat 3+ conditions
- **Centralized training, decentralized execution (CTDE)**
- **Novel wind-resilient formation control**

## Architecture
- **4 drones** with shared observation encoder
- **2-round message passing** for inter-drone communication
- **Centralized critic** for stable training
- **Discrete actions**: Stay, N, S, E, W
- **Grid world**: 15×15 (225 cells)

In [ ]:
!pip install -q gymnasium matplotlib 2>/dev/null || true
print('✅ Dependencies ready')

In [ ]:
import os, sys, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import matplotlib.pyplot as plt

PROJECT_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'swarm_grid_env.py' in files:
        PROJECT_DIR = root
        break
if not PROJECT_DIR:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'hurricane_env.py' in files:
            PROJECT_DIR = root
            break
if not PROJECT_DIR:
    raise FileNotFoundError('Could not find project files!')

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'✅ Found project: {PROJECT_DIR}')

In [ ]:
from swarm_grid_env import SwarmGridWorld, SwarmGridConfig, CurriculumSwarmGrid
from swarm_grid_model import SwarmGridModel, SwarmPPOTrainer, SwarmRolloutBuffer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', 0) / 1e9
    print(f'GPU: {props.name} | Memory: {mem:.1f} GB')

# ═══════════════════════════════════════════════════════════
# PATCH: Force device placement in update() regardless of dataset version
# This ensures tensors are on the same device as the model
# ═══════════════════════════════════════════════════════════
_orig_update = SwarmPPOTrainer.update
def _patched_update(self, obs, actions, log_probs, rewards, dones, values):
    """GPU-safe PPO update — moves all tensors to model device."""
    dev = next(self.model.parameters()).device
    obs_t = torch.FloatTensor(obs).to(dev)
    actions_t = torch.LongTensor(actions).to(dev)
    old_lp_t = torch.FloatTensor(log_probs).to(dev)
    dones_t = torch.FloatTensor(dones).to(dev)
    values_t = torch.FloatTensor(values).to(dev)
    
    advantages, returns = self.compute_gae(rewards, values, dones)
    adv_t = torch.FloatTensor(advantages).to(dev)
    ret_t = torch.FloatTensor(returns).to(dev)
    adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)
    
    T, K = obs_t.shape[:2]
    total_pl, total_vl, total_ent, n = 0, 0, 0, 0
    batch_size = T
    mini_batch = min(128, batch_size)
    
    for _ in range(4):
        idx = np.random.permutation(batch_size)
        for start in range(0, batch_size, mini_batch):
            bi = idx[start:start + mini_batch]
            batch_obs = obs_t[bi]
            batch_actions = actions_t[bi]
            batch_old_lp = old_lp_t[bi]
            batch_adv = adv_t[bi]
            batch_ret = ret_t[bi]
            
            result = self.model.evaluate_actions(batch_obs, batch_actions)
            new_lp = result['log_probs']
            values_pred = result['values']
            entropy = result['entropy']
            
            ratio = torch.exp(new_lp - batch_old_lp)
            s1 = ratio * batch_adv
            s2 = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * batch_adv
            policy_loss = -torch.min(s1, s2).mean()
            
            values_expanded = values_pred.unsqueeze(1).expand_as(batch_ret)
            value_loss = F.mse_loss(values_expanded, batch_ret)
            
            loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy.mean()
            
            self.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
            self.optimizer.step()
            
            total_pl += policy_loss.item()
            total_vl += value_loss.item()
            total_ent += entropy.mean().item()
            n += 1
    
    return {
        'policy_loss': total_pl / max(n, 1),
        'value_loss': total_vl / max(n, 1),
        'entropy': total_ent / max(n, 1),
    }

SwarmPPOTrainer.update = _patched_update
print('✅ Device patch applied — update() will work on any device')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════

TOTAL_STEPS = 150_000
ROLLOUT = 300
NUM_DRONES = 4
GRID_SIZE = 15
LR = 3e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
ENTROPY_COEF = 0.01
COMM_ROUNDS = 2
HIDDEN_DIM = 128
CKPT_DIR = '/kaggle/working/checkpoints'
CKPT_EVERY = 50

os.makedirs(CKPT_DIR, exist_ok=True)

config = SwarmGridConfig(
    grid_size=GRID_SIZE, num_drones=NUM_DRONES,
    max_steps=300, wind_prob=0.0,
)
env = CurriculumSwarmGrid(config)

# ═══════════════════════════════════════════════════════════
# FORCE PHASE 0 — always start with no wind
# The curriculum will advance naturally as training progresses
# ═══════════════════════════════════════════════════════════
env.current_phase = 0
env.phase_steps = 0
env.recent_coverages = []
env.env.config.wind_prob = 0.0

K = env.K
OBS_DIM = env.observation_space.shape[0]
ACT_DIM = env.action_space.n

model = SwarmGridModel(
    obs_dim=OBS_DIM, action_dim=ACT_DIM,
    hidden_dim=HIDDEN_DIM, num_drones=K,
    num_comm_rounds=COMM_ROUNDS,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} params ({n_params/1e6:.3f}M)')
print(f'Obs: {OBS_DIM}D | Act: {ACT_DIM}D | Drones: {K} | Grid: {GRID_SIZE}x{GRID_SIZE}')
print(f'Steps: {TOTAL_STEPS:,} | Rollout: {ROLLOUT} | lr={LR}')
print(f'\n✅ Starting at Phase 0 (0% wind) — curriculum will advance automatically')

In [ ]:
# ═══════════════════════════════════════════════════════════
# AUTO-RESUME (skip old checkpoints from previous broken runs)
# ═══════════════════════════════════════════════════════════

global_step = 0
best_reward = -float('inf')
best_coverage = 0
start_time = time.time()

log_path = '/kaggle/working/training_log.json'
# Start fresh — ignore old logs from broken runs
log = {'status': 'training', 'iterations': []}
print('🆕 Starting fresh training (ignoring old checkpoints)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# RANDOM BASELINE
# ═══════════════════════════════════════════════════════════

print('Computing random baseline (no wind)...')
env.env.config.wind_prob = 0.0
random_coverages = []
for _ in range(50):
    obs, _ = env.reset()
    for _ in range(300):
        actions = np.array([env.action_space.sample() for _ in range(K)])
        obs_all, rewards, dones, truncs, infos = env.step_all_drones(actions)
        if dones[0]: break
    random_coverages.append(infos[0]['coverage_pct'])

random_mean = np.mean(random_coverages)
print(f'Random baseline: {random_mean:.1f}% +/- {np.std(random_coverages):.1f}%')

# Reset back to Phase 0
env.current_phase = 0
env.phase_steps = 0
env.recent_coverages = []
env.env.config.wind_prob = 0.0

In [ ]:
# ═══════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════

trainer = SwarmPPOTrainer(model=model, lr=LR, gamma=GAMMA, lam=GAE_LAMBDA, entropy_coef=ENTROPY_COEF)
buffer = SwarmRolloutBuffer(num_drones=K)

print(f'\n{"="*70}')
print(f'  MARAHS: Multi-Agent Hurricane Coverage Training')
print(f'  Steps: {global_step:,} -> {TOTAL_STEPS:,}')
print(f'  Model: {n_params:,} params | {K} drones | Device: {device}')
print(f'{"="*70}\n')

it = global_step // ROLLOUT
fps_window = []

while global_step < TOTAL_STEPS:
    t0 = time.time()
    obs, _ = env.reset()
    buffer.reset()
    
    model.eval()
    for step in range(ROLLOUT):
        obs_all = env.get_all_observations()
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs_all).unsqueeze(0).to(device)
            result = model.get_actions(obs_t)
            actions = result['actions'].cpu().numpy()[0]
            log_probs = result['log_probs'].cpu().numpy()[0]
            value = result['values'].cpu().item()
        
        obs_all_new, rewards, dones, truncs, infos = env.step_all_drones(actions)
        values_arr = np.full(K, value)
        buffer.add(obs_all, actions, log_probs, rewards, dones.astype(float), values_arr)
        
        if dones[0]:
            obs, _ = env.reset()
        
        global_step += K
    
    model.train()
    data = buffer.get_all()
    metrics = trainer.update(
        obs=data['obs'], actions=data['actions'],
        log_probs=data['log_probs'], rewards=data['rewards'],
        dones=data['dones'], values=data['values'],
    )
    trainer.scheduler.step()
    
    it_t = time.time() - t0
    fps_window.append(it_t)
    if len(fps_window) > 20: fps_window.pop(0)
    avg_fps = (ROLLOUT * K) / max(np.mean(fps_window), 0.001)
    
    coverage = infos[0].get('coverage_pct', 0)
    cells = infos[0].get('cells_covered', 0)
    phase = infos[0].get('phase', 0)
    wind = infos[0].get('wind_prob', 0)
    current_lr = trainer.optimizer.param_groups[0]['lr']
    hrs_rem = (TOTAL_STEPS - global_step) / max(avg_fps, 1) / 3600
    
    if coverage > best_coverage: best_coverage = coverage
    
    log_entry = {
        'it': it, 'step': global_step,
        'cov': round(coverage, 1), 'cells': cells, 'best_cov': round(best_coverage, 1),
        'phase': phase, 'wind': wind,
        'fps': round(avg_fps, 1), 'lr': round(current_lr, 7),
        'entropy': round(metrics['entropy'], 4),
    }
    log['iterations'].append(log_entry)
    
    print(f'it {it:5d} | step {global_step:>9,} | '
          f'cov {coverage:5.1f}% ({cells:3d}/225) | best {best_coverage:.0f}% | '
          f'ph {phase} | wind {wind:.0%} | '
          f'{avg_fps:.0f}fps | lr={current_lr:.1e} | '
          f'ent={metrics["entropy"]:.3f} | ETA {hrs_rem:.1f}h')
    
    if it % CKPT_EVERY == 0 and it > 0:
        torch.save({
            'global_step': global_step, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': trainer.optimizer.state_dict(),
            'best_reward': best_reward, 'best_coverage': best_coverage,
        }, f'{CKPT_DIR}/ckpt_{global_step}.pt')
        torch.save({
            'global_step': global_step, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': trainer.optimizer.state_dict(),
            'best_reward': best_reward, 'best_coverage': best_coverage,
        }, f'{CKPT_DIR}/latest.pt')
        log['global_step'] = global_step
        with open(log_path, 'w') as f:
            json.dump(log, f)
        print(f'    -> checkpoint saved (best={best_coverage:.0f}%)')
    
    it += 1

# FINAL SAVE
torch.save({
    'global_step': global_step, 'model_state_dict': model.state_dict(),
    'optimizer_state_dict': trainer.optimizer.state_dict(),
    'best_reward': best_reward, 'best_coverage': best_coverage,
}, f'{CKPT_DIR}/best_model.pt')

log['status'] = 'complete'
log['final_step'] = global_step
log['training_time_s'] = round(time.time() - start_time, 1)
log['best_coverage'] = best_coverage
with open(log_path, 'w') as f:
    json.dump(log, f)

print(f'\n{"="*70}')
print(f'  ✅ Training complete! Steps: {global_step:,}')
print(f'  Best coverage: {best_coverage:.1f}% (random baseline: {random_mean:.1f}%)')
print(f'  Improvement: {best_coverage - random_mean:+.1f}% over random')
print(f'{"="*70}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# VISUALIZATION
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Learning curve
if len(log['iterations']) > 0:
    its = [e['it'] for e in log['iterations']]
    covs = [e['cov'] for e in log['iterations']]
    bests = [e['best_cov'] for e in log['iterations']]
    axes[0, 0].plot(its, covs, 'b-', alpha=0.5, label='Per-iteration')
    axes[0, 0].plot(its, bests, 'r-', linewidth=2, label='Best so far')
    axes[0, 0].axhline(y=random_mean, color='gray', linestyle='--', label=f'Random ({random_mean:.0f}%)')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Coverage %')
    axes[0, 0].set_title('Training Progress')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

# Bar chart
methods = ['Random\nBaseline', '4-Drone\nSwarm (AI)']
coverages_bar = [random_mean, best_coverage]
colors = ['#ff6b6b', '#45b7d1']
bars = axes[0, 1].bar(methods, coverages_bar, color=colors, width=0.5)
axes[0, 1].set_ylabel('Coverage %')
axes[0, 1].set_title('Random vs Trained Swarm')
axes[0, 1].set_ylim(0, 105)
for bar, cov in zip(bars, coverages_bar):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                    f'{cov:.0f}%', ha='center', fontweight='bold', fontsize=14)

# Coverage distribution
axes[1, 0].hist(random_coverages, bins=15, alpha=0.6, label='Random', color='red')
axes[1, 0].axvline(x=best_coverage, color='blue', linewidth=2, label=f'Trained Best ({best_coverage:.0f}%)')
axes[1, 0].set_xlabel('Coverage %')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Coverage Distribution')
axes[1, 0].legend()

# Phase progression
if len(log['iterations']) > 0:
    phases = [e['phase'] for e in log['iterations']]
    axes[1, 1].plot(its, phases, 'g-', linewidth=2)
    axes[1, 1].set_xlabel('Iteration')
    axes[1, 1].set_ylabel('Curriculum Phase')
    axes[1, 1].set_title('Curriculum Progression')
    axes[1, 1].set_yticks(range(len(env.phases)))
    axes[1, 1].set_yticklabels([f'{p:.0%} wind' for p in env.phases])
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/marahs_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved!')